In [18]:
import torch_geometric
from torch_geometric.nn.aggr import Aggregation, SumAggregation
from torch_geometric.datasets import Planetoid

In [2]:
dataset = Planetoid(root="tutorial1",name= "Cora")

Processing...
Done!


In [3]:
print(dataset)
print("number of graphs:\t\t",len(dataset))
print("number of classes:\t\t",dataset.num_classes)
print("number of node features:\t",dataset.num_node_features)
print("number of edge features:\t",dataset.num_edge_features)

Cora()
number of graphs:		 1
number of classes:		 7
number of node features:	 1433
number of edge features:	 0


In [4]:
import os.path as osp

import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

In [5]:
data = dataset[0]

In [13]:
data.x = data.x.to_sparse()
data.train_mask = data.train_mask.to_sparse()
data.val_mask = data.val_mask.to_sparse()
data.test_mask = data.test_mask.to_sparse()
data.edge_index = data.edge_index.to_sparse()

In [25]:
class Net(torch.nn.Module):
    def __init__(self, aggr):
        super(Net, self).__init__()
        
        self.conv = SAGEConv(dataset.num_features,
                             dataset.num_classes,
                             aggr=aggr) # max, mean, add ...)

    def forward(self, x):
        x = self.conv(data.x, data.edge_index)
        return F.log_softmax(x, dim=1)

In [26]:
help(SAGEConv)

Help on class SAGEConv in module torch_geometric.nn.conv.sage_conv:

class SAGEConv(torch_geometric.nn.conv.message_passing.MessagePassing)
 |  SAGEConv(in_channels: Union[int, Tuple[int, int]], out_channels: int, aggr: Union[str, List[str], torch_geometric.nn.aggr.base.Aggregation, NoneType] = 'mean', normalize: bool = False, root_weight: bool = True, project: bool = False, bias: bool = True, **kwargs)
 |  
 |  The GraphSAGE operator from the `"Inductive Representation Learning on
 |  Large Graphs" <https://arxiv.org/abs/1706.02216>`_ paper.
 |  
 |  .. math::
 |      \mathbf{x}^{\prime}_i = \mathbf{W}_1 \mathbf{x}_i + \mathbf{W}_2 \cdot
 |      \mathrm{mean}_{j \in \mathcal{N(i)}} \mathbf{x}_j
 |  
 |  If :obj:`project = True`, then :math:`\mathbf{x}_j` will first get
 |  projected via
 |  
 |  .. math::
 |      \mathbf{x}_j \leftarrow \sigma ( \mathbf{W}_3 \mathbf{x}_j +
 |      \mathbf{b})
 |  
 |  as described in Eq. (3) of the paper.
 |  
 |  Args:
 |      in_channels (int or tup

In [36]:
class COOMaxAggregation(Aggregation):
    r"""An aggregation operator that takes the feature-wise maximum across a
    set of elements.

    .. math::
        \mathrm{max}(\mathcal{X}) = \max_{\mathbf{x}_i \in \mathcal{X}}
        \mathbf{x}_i.
    """
    def forward(self, x, index = None,
                ptr = None, dim=-2, dim_size = None):
#         self.reduce(x, index, ptr, dim_size, dim, reduce='max)
        
        
        return torch_scatter.segment_coo(x, index, dim, dim_size=dim_size,
                                     reduce='max')

In [37]:
aggr = COOMaxAggregation()

In [38]:
device = torch.device('cuda')
model, data = Net(aggr=aggr).to(device), data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [39]:
model(data.x)

ValueError: Encountered tensor with size 2708 in dimension -2, but expected size 10556.

In [40]:
from torch_scatter import segment_coo

src = torch.randn(10, 6, 64)
index = torch.tensor([0, 0, 1, 1, 1, 2])
index = index.view(1, -1)  # Broadcasting in the first and last dim.

out = segment_coo(src, index, reduce="sum")

print(out.size())

torch.Size([10, 3, 64])


In [41]:
index.shape

torch.Size([1, 6])

In [42]:
src.shape

torch.Size([10, 6, 64])